# Mi publicación de una entidad GTFS

Este notebook es la entrada del Job. Valida y publica una entidad y un snapshot en una ejecución, usando las reglas ya probadas. Tengo habilitados los procesadores agency, routes y stops. Routes requiere agency del mismo snapshot; stops valida su jerarquía dentro de su propia tabla. La disponibilidad del procesador no demuestra su publicación en el workspace.

**Efectos en mode=publish:** crea o actualiza Silver, cuarentena y auditoría. Con mode=validate solo calcula el resultado, sin escribir. No necesito ejecutar antes 08 o 09 cada vez: eran herramientas de aprendizaje y diagnóstico.

Configuro las dependencias de requirements-silver.txt en el entorno del notebook o del Job antes de ejecutar. Evito instalar y reiniciar Python dentro de cada iteración. Necesito el repositorio completo, no solo importar este notebook.

El resumen del bloque 2 incluye los conteos verificados después de publicar. Evito consultar de nuevo toda la historia en cada iteración; puedo hacerlo de forma puntual con el SQL del runbook.


## Bloque 1 — Elegir la ejecución

Los defaults corresponden a mi entorno y a 2025. La publicación usa versión + hash. Conservaré 2025 cuando publique 2026. Una sola ejecución escritora a la vez evita carreras entre tareas manuales y Jobs.


In [ ]:
# BLOQUE 1 — Código y parámetros de ejecución
from pathlib import Path
import json
import sys

defaults = {
    "repo_root": "", "catalog": "workspace", "schema": "gtfs_bogota",
    "staging_volume": "staging_gtfs", "entity": "agency",
    "feed_version": "2025-10-28",
    "snapshot_sha256": "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "mode": "publish", "verify_history": "true",
}
for name, default in defaults.items():
    dbutils.widgets.text(name, default)
parameters = {name: dbutils.widgets.get(name).strip() for name in defaults}
root_parameter = parameters.pop("repo_root")
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter).resolve()] if root_parameter else [current, *current.parents]
project_root = next((root for root in roots if (root / "databricks/gtfs_silver/pipeline.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)
from gtfs_silver.pipeline import run_entity
if parameters["verify_history"] not in {"true", "false"}:
    raise ValueError("verify_history debe ser true o false.")
parameters["verify_history"] = parameters["verify_history"] == "true"
print(f"Entidad: {parameters['entity']}; versión: {parameters['feed_version']}; modo: {parameters['mode']}")


## Bloque 2 — Validar y publicar en una sola llamada

La llamada lee Bronze y el encabezado, valida, registra el inicio y los rechazos, publica solo el snapshot válido y verifica su lectura posterior. La auditoría termina en SUCCEEDED, REJECTED o FAILED. Si falla una etapa previa a tener un resultado de validación, el error queda en la ejecución del Job.

Con verify_history=true compararé las otras versiones contra una lectura Delta fijada antes de la escritura. Es razonable para agency, routes y stops actuales; no impondré esa lectura completa a entidades grandes sin medir su costo.


In [ ]:
# BLOQUE 2 — Ejecución completa por entidad y snapshot
publication_result = run_entity(spark, **parameters)
print(json.dumps(publication_result, ensure_ascii=False, indent=2))
